In [ ]:
import os
from glob import glob
from os.path import join as pjoin
import nibabel as nb
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.io

In [ ]:
# base_dir = pjoin(os.getenv('HOME'), 'sleepstudy')

# loading ROI file for pattern analysis (can be any ROI in functional resolution)

base_dir = '/mnt/tambinidata/sleepstudy'
deriv_dir = pjoin(base_dir, 'data', 'derivatives')

roi = glob(pjoin(base_dir, 'analysis', 'fusi_letter_ROI*'))
roi_file = roi[0]

roi_data = nb.load(roi_file).get_data()
np.sum(roi_data==1)

In [ ]:
ss_list = [105, 106, 108, 111, 112, 113, 114, 115, 116, 118, 121, 123, 124, 125, 126, 127, \
            130, 131, 132, 133, 135, 136, 137, 138, 141, 142, 144, 148, 149, 152, 154, 156, 157]
# ss_list = [105, 106]
Nitems = 7 # looping over items in each OSPAN block
Nblocks = 8 # number of blocks per scan
Nses = 4

# similarity between the same item (serial position) during encoding & retrieval (same_sim)
same_sim = np.full((len(ss_list), Nses), np.nan)
# off-diagonal similarities
other_sim = np.full((len(ss_list), Nses), np.nan)

# df_sim = pd.DataFrame(columns=['sub', 'ses', 'sim-same', 'sim-diff'])
df_sim = pd.DataFrame(columns=['sub', 'ses', 'sim', 'type'])
df = []

# behav data info
beh_dir = pjoin(base_dir, 'analysis', 'os_correct_trials_index')

for iss, ss in enumerate(ss_list):
    # print(ss)
    ss_dir = pjoin(deriv_dir, 'fmriprep', 'sub-' + str(ss))

    ss_mask = glob(pjoin(ss_dir, '*combined-mask*'))
    assert(len(ss_mask)==1)
    mask = nb.load(ss_mask[0]).get_data()==1

    # roi data & mask combined
    roi_curr = np.logical_and(mask==1, roi_data==1)

    # ses information
    if ss==127 or ss==136:
        ses_list = ['ses-1', 'ses-2']
        ses_dirs = []
        for ses in ses_list:
            ses_dirs.append(pjoin(ss_dir, ses))
    else:
        ses_dirs = glob(pjoin(ss_dir, 'ses-*'))

    if len(ses_dirs)==2:
        print(ss)
        continue

    # loop over sessions
    for ises,ses_dir in enumerate(ses_dirs):

        # load in relevant behavioral file

        spm_dir = pjoin(ses_dir, 'spm_os_model_items')
        
        # correlation matrix - 7x7x8
        sim = np.zeros((Nitems, Nitems, Nblocks))

        for iblock in range(0, Nblocks):

            # 7 items in each encoding block - starts at 0004 since 1-3 are regressors for different blocks
            e_file_range = np.arange(4+(iblock*Nitems), 4+(iblock*Nitems)+Nitems)

            # loop over each retrieval item
            for iritem in range(0, Nitems):
                r_file_n = 60 + iritem + (iblock*Nitems) # where retrieval files begin (after encoding)
                if r_file_n < 100:
                    r_file_n = 'beta_00' + str(r_file_n)
                elif r_file_n < 1000:
                    r_file_n = 'beta_0' + str(r_file_n)

                # filename for retrieval item
                r_filename = glob(pjoin(spm_dir, r_file_n + '*'))
                assert(len(r_filename)==1)

                # load retrieval item
                r_img = nb.load(r_filename[0])
                r_data = r_img.get_data()[roi_curr==1] # grab data for ROI only

                # loop over each item in encoding
                for iefile, e_file_n in enumerate(e_file_range):
                    if e_file_n < 10:
                        e_file = 'beta_000' + str(e_file_n)
                    elif e_file_n < 100:
                        e_file = 'beta_00' + str(e_file_n)
                    
                    e_filename = glob(pjoin(spm_dir, e_file + '*'))

                    assert(len(e_filename)==1)
                    # load in encoding data, fro that ROI
                    e_img = nb.load(e_filename[0])
                    e_data = e_img.get_data()[roi_curr==1]

                    # compute similarity between encoding item & retrieval item
                    out = scipy.stats.pearsonr(r_data, e_data)
                    sim[iritem, iefile, iblock] = np.arctanh(out[0]) # arctanh = fisher z-transform of correlation value

                    # if encoding/retrieval pair are both accurate - give a value of 1
                    # if encoding/retrieval pair are both inaccurate - give a value of 0
                    # if encoding/retrieval pair are mixed - give a value of X


        # select the similarity for accuracy==1 trial pairs
        # new_sim = sim[acc==1] # might vectorize


        # taking mean similarity across 8 blocks
        sim_mn = np.nanmean(sim, axis=2)

        # extracting diagonal values from matrix, averaging
        same_sim[iss, ises] = np.nanmean(np.diag(sim_mn))

        # non-diagonal parts of matrix
        temp = np.tril(sim_mn)
        lower = temp[temp!=0]
        temp = np.triu(sim_mn)
        upper = temp[temp!=0]
        other_sim[iss, ises] = np.nanmean([lower, upper])

        if 'ses-1' in ses_dir:
            ses_label = 'pm'
        elif 'ses-3' in ses_dir:
            ses_label = 'pm'
        elif 'ses-2' in ses_dir:
            ses_label = 'am'
        elif 'ses-4' in ses_dir:
            ses_label = 'am'
        # temp = pd.DataFrame({'sub': ss, 'ses': ses_label, 'sim-same': same_sim[iss, ises], 'sim-diff': other_sim[iss, ises]}, index=[0])
        temp_1 = pd.DataFrame({'sub': ss, 'ses': ses_label, 'sim': same_sim[iss, ises], 'type': 'match'}, index=[0])
        temp_2 = pd.DataFrame({'sub': ss, 'ses': ses_label, 'sim': other_sim[iss, ises], 'type': 'nonmatch'}, index=[0])
        temp_3 = pd.DataFrame({'sub': ss, 'ses': ses_label, 'sim': same_sim[iss, ises] - other_sim[iss, ises], 'type': 'diff'}, index=[0])
        # df_sim = df_sim.append(temp_1)
        # df_sim = df_sim.append(temp_2)
        # df_sim = df_sim.append(temp_3)
        df.append(temp_1)        
        df.append(temp_2)        
        df.append(temp_3)

In [ ]:
df_sim = pd.concat(df)
df_sim

In [ ]:
sns.barplot(data=df_sim, x='ses', y='sim', ci=68, hue='type')

In [ ]:
diff = same_sim-other_sim
print(np.nanmean(diff, axis=0))
diff_pm = np.mean(np.vstack((diff[:, 0], diff[:,2])),axis=0)
diff_am = np.mean(np.vstack((diff[:, 1], diff[:,3])),axis=0)
diff_comb = np.vstack((diff_am, diff_pm)).T
diff_comb.shape

In [ ]:
sns.barplot(data=diff_comb, ci=68)
plt.figure()
sns.barplot(data=diff, ci=68)

In [ ]:
scipy.stats.ttest_1samp(diff_am, popmean=0, nan_policy='omit')

In [ ]:
scipy.stats.ttest_ind(diff_am, diff_pm, nan_policy='omit')

In [ ]:
am_diff = df_sim.query("type=='diff' and ses=='am'")
pm_diff = df_sim.query("type=='diff' and ses=='pm'")
print(am_diff.shape)
scipy.stats.ttest_1samp(am_diff['sim']-pm_diff['sim'], popmean=0)

In [ ]:
diff_pm

In [ ]:
ss = 'sub-106'
ses = 'ses-1'

spm_dir = pjoin(deriv_dir, 'fmriprep', ss, ses, 'spm_os_model_items')

In [ ]:
Nitems = 7
Nblocks = 8

sim = np.zeros((Nitems, Nitems, Nblocks))
# iblock =7 # 8 blocks total
# iitem = 6 # 7 items per block
# e_file_n = 4 + iitem + (iblock*7)

for iblock in range(0, Nblocks):
# e_file_range = np.setdiff1d(np.arange(4+(iblock*7), 4+(iblock*7)+7), e_file_n)
    e_file_range = np.arange(4+(iblock*7), 4+(iblock*7)+7)
    print(iblock)

# if e_file_n < 10:
#     e_file = 'beta_000' + str(e_file_n)
# elif e_file_n < 100:
#     e_file = 'beta_00' + str(e_file_n)

    for iritem in range(0, Nitems):
        # print(iritem)
        r_file_n = 60 + iritem + (iblock*7)
        if r_file_n < 100:
            r_file_n = 'beta_00' + str(r_file_n)
        elif r_file_n < 1000:
            r_file_n = 'beta_0' + str(r_file_n)

        r_filename = glob(pjoin(spm_dir, r_file_n + '*'))
        assert(len(r_filename)==1)
        r_img = nb.load(r_filename[0])
        r_data = r_img.get_data()[roi_data==1]

        for ifile, e_file_n in enumerate(e_file_range):
            if e_file_n < 10:
                e_file = 'beta_000' + str(e_file_n)
            elif e_file_n < 100:
                e_file = 'beta_00' + str(e_file_n)
            
            e_filename = glob(pjoin(spm_dir, e_file + '*'))

            assert(len(e_filename)==1)
            e_img = nb.load(e_filename[0])
            e_data = e_img.get_data()[roi_data==1]
            out = scipy.stats.pearsonr(r_data, e_data)
            sim[iritem, ifile, iblock] = out[0]

sim_mn = np.mean(sim, axis=2)
same_sim[iss, ioth] = np.mean(np.diag(sim_mn))

temp = np.tril(sim_mn)
lower = temp[temp!=0]
temp = np.triu(sim_mn)
upper = temp[temp!=0]
other_sim[iss, ioth] = np.mean([lower, upper])



In [ ]:
sim_mn = np.mean(sim, axis=2)
plt.imshow(sim_mn)
plt.colorbar()
sim[0,-1,:]

In [ ]:
for ifile, e_file_n in enumerate(e_file_range):
    if e_file_n < 10:
        e_file = 'beta_000' + str(e_file_n)
    elif e_file_n < 100:
        e_file = 'beta_00' + str(e_file_n)
    
    e_filename = glob(pjoin(spm_dir, e_file + '*'))

    assert(len(e_filename)==1)
    e_img = nb.load(e_filename[0])
    e_data = e_img.get_data()[roi_data==1]
    out = scipy.stats.pearsonr(r_data, e_data)
    sim[ifile] = out[0]

In [ ]:
np.mean(np.diag(sim_mn))

In [ ]:
temp = np.tril(sim_mn)
lower = temp[temp!=0]
temp = np.triu(sim_mn)
upper = temp[temp!=0]
other = np.mean([lower, upper])
other